In [1]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import evaluate
from transformers import BlipProcessor, BlipForConditionalGeneration

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT_DIR = "../data_ready_for_kaggle"
TEST_PATH = os.path.join(ROOT_DIR, "cleaned_test.csv")
IMAGES_DIR = os.path.join(ROOT_DIR, "images_resized")
MODEL_DIR = os.path.abspath("../saved_models_v3")
SAVE_DIR = "./model_quantized/pytorch_int8"
RESULTS_DIR = "./results"
MAX_TEST_IMAGES = 200
MAX_LENGTH = 64
BATCH_SIZE = 16
NUM_WORKERS = 0
NUM_BEAMS = 5
NO_REPEAT_NGRAM_SIZE = 3
REPETITION_PENALTY = 1.5
WARMUP_STEPS = 10
device = torch.device("cpu")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Device: {device}")
print(f"Model dir: {MODEL_DIR}")

Device: cpu
Model dir: c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\saved_models_v3


In [3]:
class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=64):
        self.df = pd.read_csv(data_file)
        self.img_dir = img_dir
        self.processor = processor
        self.max_length = max_length
        self.image_groups = self.df.groupby('image_file')['caption'].apply(list).reset_index()

    def __len__(self):
        return len(self.image_groups)

    def __getitem__(self, index):
        row = self.image_groups.iloc[index]
        image_file = row['image_file']
        caption = row['caption'][0]
        image = Image.open(os.path.join(self.img_dir, image_file)).convert('RGB')

        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding['image_file'] = image_file
        encoding['raw_captions'] = row['caption']
        return encoding

def collate_fn(batch):
    result = {}
    for key in ['pixel_values', 'input_ids', 'attention_mask']:
        result[key] = torch.stack([item[key] for item in batch])
    result['image_file'] = [item['image_file'] for item in batch]
    result['raw_captions'] = [item['raw_captions'] for item in batch]
    return result


In [4]:
processor = BlipProcessor.from_pretrained(MODEL_DIR)
model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
model.eval()

quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)
quantized_model.eval()

torch.save(quantized_model.state_dict(), os.path.join(SAVE_DIR, 'pytorch_int8_state_dict.pt'))
processor.save_pretrained(SAVE_DIR)
print(f"Saved quantized artifacts at: {SAVE_DIR}")


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 3622.49it/s]
C:\Users\nviquang\AppData\Local\Temp\ipykernel_2024\3743522388.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Saved quantized artifacts at: ./model_quantized/pytorch_int8


In [5]:
test_dataset = UITViICDataset(TEST_PATH, IMAGES_DIR, processor, max_length=MAX_LENGTH)
test_dataset.image_groups = test_dataset.image_groups.head(MAX_TEST_IMAGES).reset_index(drop=True)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

print(f"Test images: {len(test_dataset)}")
print(f"Test batches: {len(test_dataloader)}")

def build_multi_references(dataloader):
    references = {}
    for batch in dataloader:
        for image_file, raw_captions in zip(batch['image_file'], batch['raw_captions']):
            references[image_file] = raw_captions
    return references

def generate_captions(model, processor, dataloader, device):
    model.eval()
    results = {}
    latencies = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc='Generating INT8 captions')):
            pixel_values = batch['pixel_values'].to(device)

            start_time = time.perf_counter()
            generated_ids = model.generate(
                pixel_values=pixel_values,
                max_length=MAX_LENGTH,
                num_beams=NUM_BEAMS,
                early_stopping=True,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                repetition_penalty=REPETITION_PENALTY,
            )
            end_time = time.perf_counter()

            if batch_idx >= WARMUP_STEPS:
                latencies.append(end_time - start_time)

            preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
            for image_file, pred in zip(batch['image_file'], preds):
                results[image_file] = pred

    return results, latencies

def calculate_metrics(predictions_dict, references_dict):
    common_images = sorted(set(predictions_dict.keys()) & set(references_dict.keys()))
    predictions = [predictions_dict[img] for img in common_images]
    references = [references_dict[img] for img in common_images]

    bleu_metric = evaluate.load('bleu')
    rouge_metric = evaluate.load('rouge')
    meteor_metric = evaluate.load('meteor')

    bleu_results = bleu_metric.compute(predictions=predictions, references=references)
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    meteor_results = meteor_metric.compute(predictions=predictions, references=references)

    return {
        'bleu': bleu_results,
        'rouge': rouge_results,
        'meteor': meteor_results,
        'num_images': len(common_images),
    }

def summarize_benchmark(name, predictions_dict, references_dict, latencies, backend, precision, device, provider=None):
    metrics = calculate_metrics(predictions_dict, references_dict)

    avg_latency = float(np.mean(latencies)) if latencies else 0.0
    p95_latency = float(np.percentile(latencies, 95)) if latencies else 0.0
    throughput = float(BATCH_SIZE / avg_latency) if avg_latency > 0 else 0.0

    benchmark = {
        'name': name,
        'backend': backend,
        'precision': precision,
        'device': device,
        'provider': provider,
        'batch_size': BATCH_SIZE,
        'max_test_images': MAX_TEST_IMAGES,
        'avg_latency_seconds_per_batch': avg_latency,
        'p95_latency_seconds_per_batch': p95_latency,
        'throughput_images_per_second': throughput,
        'bleu4': metrics['bleu']['bleu'] * 100,
        'rougeL': metrics['rouge']['rougeL'] * 100,
        'meteor': metrics['meteor']['meteor'] * 100,
        'num_images': metrics['num_images'],
    }
    return benchmark, metrics

Test images: 200
Test batches: 13


In [6]:
references_dict = build_multi_references(test_dataloader)
predictions_dict, latencies = generate_captions(quantized_model, processor, test_dataloader, device)
benchmark, metrics = summarize_benchmark(
    'pytorch_int8',
    predictions_dict,
    references_dict,
    latencies,
    backend='pytorch',
    precision='int8',
    device=device.type,
    provider='torch'
)

print(json.dumps(benchmark, indent=2, ensure_ascii=False))

with open(os.path.join(RESULTS_DIR, 'pytorch_int8.json'), 'w', encoding='utf-8') as f:
    json.dump(benchmark, f, ensure_ascii=False, indent=2)

Generating INT8 captions: 100%|██████████| 13/13 [09:15<00:00, 42.73s/it]
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


{
  "name": "pytorch_int8",
  "backend": "pytorch",
  "precision": "int8",
  "device": "cpu",
  "provider": "torch",
  "batch_size": 16,
  "max_test_images": 200,
  "avg_latency_seconds_per_batch": 30.92886766666682,
  "p95_latency_seconds_per_batch": 39.43903180999914,
  "throughput_images_per_second": 0.5173160612421576,
  "bleu4": 20.151105795164927,
  "rougeL": 47.639633506017034,
  "meteor": 34.002983101730344,
  "num_images": 200
}
